# Gradient Boosting — Performance with Discipline (and Leakage Avoidance)

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb13_gradient_boosting.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain how gradient boosting reduces **bias** by training trees sequentially on the residuals of the previous trees, on both classification and regression cases.
2. Fit `GradientBoostingClassifier` and `GradientBoostingRegressor`; demonstrate the CV-score lift over the random forest from nb12 and the **Week-2 reference**.
3. Diagnose the `learning_rate × n_estimators` trade-off — why "lots of trees, slow learning rate" beats "few trees, fast learning rate" at the same total fit budget.
4. Use **`staged_predict`** to plot train vs validation loss per iteration and identify the early-stopping point.
5. Recognize and prevent **boosting's amplification of leaky features** — the failure mode that nb09's leakage case studies set up.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one per case. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Random forests reduce **variance** by averaging many trees. Gradient boosting reduces **bias** by stacking trees sequentially — each new tree fits the *residuals* (regression) or *misclassifications* (classification) of the running ensemble. The two strategies attack different parts of the bias/variance decomposition; combining their lessons in nb14's selection ceremony usually produces the strongest model on tabular data.

For the **State Health Department's** screening pipeline, the case for gradient boosting is the same as for the random forest — but with one extra ask: *"can we squeeze the last point of ROC-AUC out of the model?"* That last point matters when the false-negative cost is a missed cancer diagnosis. Boosting often delivers it.

For **HomeValue Analytics'** price-prediction model, the case is sharper. The random forest in nb12 already beat the OLS reference by ~20 R² points on California Housing; gradient boosting typically lifts that by another 3–5 points, which translates to ~USD 5–8K reduction in prediction RMSE — meaningful when the median home value being predicted is around USD 200K.

The cost is real: gradient boosting is sequential (cannot fully parallelize), more sensitive to hyperparameters than random forests, and prone to overfitting if you let it run too many iterations. All three risks have specific mitigations covered in this notebook: tune `learning_rate × n_estimators` together, use `staged_predict` to identify the early-stopping point, and **never let gradient boosting see a leaky feature** — boosting will amplify the leak more aggressively than any other algorithm in this course.

> **A question that often comes up here:** *"if boosting is better than forests on tabular data, why not skip forests entirely?"* Three answers. First, forests give you OOB scoring and parallel fits — properties boosting lacks. Second, the four-method importance heatmap from nb12 transfers cleanly to boosting, but the *interpretation* of MDI is harder for boosting (sequential trees mean the importance accumulates differently). Third, the right answer is not "always boost" but "compare both under nb14's CI-overlap discipline and pick the simpler model when CIs overlap." Today's job is to add gradient boosting to the candidate roster — not to crown it.

---

## 1. Setup — Imports, References, Helpers

Same toolkit as nb12: five plot helpers (`plot_train_val_curve`, `plot_predicted_vs_actual`, `plot_cv_ci`, `plot_importance_bars`, `plot_importance_heatmap`) plus the two Week-2 reference pipelines. `staged_predict` is built into sklearn's `GradientBoostingClassifier` / `GradientBoostingRegressor` — no extra import needed.

> 💡 **Gemini Prompt:** "Set up imports for sklearn GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor, DecisionTreeClassifier, DecisionTreeRegressor, LogisticRegression, LinearRegression, log_loss, mean_squared_error, train_test_split, cross_val_score, StratifiedKFold, KFold, load_breast_cancer, fetch_california_housing, StandardScaler, Pipeline. Set RANDOM_SEED = 474. Define reference_clf and reference_reg as the Week-2 baseline pipelines. Define helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci."
>
> **After running, verify:**
> - [ ] `RANDOM_SEED = 474`, `reference_clf`, `reference_reg` defined
> - [ ] All five plot helpers callable
> - [ ] No import errors


In [ ]:
# Setup — imports, seed, Week-2 references, plot helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss, mean_squared_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'

reference_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))
])
reference_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

def plot_train_val_curve(x_values, train, val_mean, val_std, xlabel, ylabel, title, ax,
                         color_train=GREY, color_val=CLF_COLOR):
    xs = list(range(len(x_values)))
    ax.plot(xs, train, marker='o', label='Train', linewidth=2, color=color_train)
    ax.errorbar(xs, val_mean, yerr=val_std, marker='s', label='CV ± SD',
                linewidth=2, capsize=5, color=color_val)
    ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in x_values])
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

def plot_predicted_vs_actual(y_true, y_pred, ax, title='Predicted vs Actual', color=REG_COLOR):
    ax.scatter(y_true, y_pred, alpha=0.25, s=8, color=color)
    lo, hi = float(min(np.min(y_true), np.min(y_pred))), float(max(np.max(y_true), np.max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5):
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    ax.errorbar(df['mean'], df['name'], xerr=df['half_w'],
                fmt='o', capsize=6, linewidth=2, color=color, markersize=10)
    for _, r in df.iterrows():
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                r['name'], f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

print(f"✓ RANDOM_SEED = {RANDOM_SEED}")
print(f"✓ Week-2 references: reference_clf, reference_reg")
print(f"✓ Helpers: plot_train_val_curve, plot_predicted_vs_actual, plot_cv_ci")


---

## 2. Load Both Datasets

Same 60/20/20 locking discipline as nb11 and nb12. Both business cases carve the rows once into 60% training, 20% validation, and 20% test, with `random_state=RANDOM_SEED` keeping the partition identical to nb11–nb12. Every fit, every diagnostic, and every tuning decision between here and nb14 lives on the training set only; the test envelopes stay sealed.

The section is short on purpose — its job is to make `X_train_clf`, `X_train_reg`, `X_val_clf`, `X_val_reg`, `cv_clf`, and `cv_reg` available to every section that follows, so the CV scores you compute today are directly comparable to the nb11 and nb12 numbers without an asterisk.

> 💡 **Gemini Prompt:** "Apply the course's 60/20/20 split to both datasets. Load `load_breast_cancer(as_frame=True)` and `fetch_california_housing(as_frame=True)`. For each: first carve off 20% as `X_test_*` / `y_test_*` with `test_size=0.20`, `random_state=474`, and `stratify=y` for classification only; then split the remaining 80% with `test_size=0.25` into `X_train_*` / `X_val_*`. Build `cv_clf = StratifiedKFold(5, shuffle=True, random_state=474)` and `cv_reg = KFold(5, shuffle=True, random_state=474)`. Print the per-case row counts and a `[LOCKED until nb14]` marker on the test counts."
>
> **After running, verify:**
> - [ ] Classification: train ~341, val ~114, test ~114 (out of 569)
> - [ ] Regression: train ~12,384, val ~4,128, test ~4,128 (out of 20,640)
> - [ ] All splits use `random_state=RANDOM_SEED`
> - [ ] Both test-set lines include `[LOCKED until nb14]`


In [ ]:
# 60/20/20 split — identical partition to nb11 and nb12 under RANDOM_SEED.
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target
X_clf_temp, X_test_clf, y_clf_temp, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
X_train_clf, X_val_clf, y_train_clf, y_val_clf = train_test_split(
    X_clf_temp, y_clf_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_clf_temp
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target
X_reg_temp, X_test_reg, y_reg_temp, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(
    X_reg_temp, y_reg_temp, test_size=0.25, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification — Train: {len(X_train_clf):>5} | Val: {len(X_val_clf):>4} | Test: {len(X_test_clf):>4} [LOCKED until nb14]')
print(f'Regression     — Train: {len(X_train_reg):>5} | Val: {len(X_val_reg):>4} | Test: {len(X_test_reg):>4} [LOCKED until nb14]')


**Reading the output:**

Three holdout splits, two locked envelopes, two CV plans — same as nb11 and nb12. The classification training set has about **341 patients** and the regression training set has about **12,384 tracts** (a 36× size gap that has shaped every modelling decision since nb11). `StratifiedKFold` preserves the ~63% benign / ~37% malignant ratio on the classification side; plain `KFold` works on the regression side because the target is continuous. The `X_val_*` sets are reserved for one-shot held-out checks — Section 7 will use them for the `staged_predict` early-stopping curve — and the `X_test_*` sets stay sealed until nb14's selection ceremony.

> **A question that often comes up here:** *"my Section 4 numbers will barely change from nb12's — is that a problem?"* No — it is the point. nb11, nb12, nb13, and nb14 all use exactly the same `RANDOM_SEED=474` and the same 60/20/20 procedure, so the four notebooks operate on **identical** training, validation, and test sets. The Week-2 reference and random-forest scores in nb13 §4 will match nb12's to four decimal places. That consistency is the discipline that makes nb14's selection ceremony defensible.

**Key takeaway.** Same splits, same seed, same CV folds across nb11 → nb14. Every CV mean in nb13 is directly comparable to the nb11 and nb12 numbers, and the test envelopes stay sealed until nb14.

---

## 3. Boosting vs Bagging — Sequential vs Parallel

nb12 built up the **bagging** recipe: train many trees in parallel on bootstrap samples, then average their predictions. Today's gradient boosting uses the opposite recipe — **boosting**: train one tree, look at where the ensemble is still wrong, train the next tree specifically to fix those mistakes, and keep going. The schematic below contrasts the two side-by-side, because the *shape* of the algorithm matters before any code runs.

Think of the contrast in business terms. Bagging is **a panel of independent analysts** — each one looks at a slightly different slice of the data, hands in their own forecast, and the firm publishes the average across the panel. The analysts do not know about each other; the averaging is what cancels out individual noise. Boosting is **an apprentice-and-master chain** — analyst 1 hands in a forecast, analyst 2 looks at where 1 was wrong and adds a small correction, analyst 3 looks at where 1 and 2 *together* are still wrong and adds another correction, and so on for hundreds of iterations. Each new analyst is *deliberately specialized* on the running team's leftover error. That sequential structure is what makes boosting reduce **bias** (systematic mistakes the ensemble keeps making) rather than variance.

> 💡 **Gemini Prompt:** "Render a side-by-side schematic comparing bagging (left) and boosting (right). Bagging panel: draw five parallel arrows from a central training-data block, each ending at a tree box, with a final averaging step on the right labeled 'Average'. Boosting panel: draw a chain — Tree 1 fits the data, an arrow flows from Tree 1 to a 'Residuals' box, then to Tree 2, then to another 'Residuals' box, then to Tree 3, ending with a final 'Weighted sum' box. Label the left panel 'Bagging — parallel, order-independent (variance ↓)' and the right panel 'Boosting — sequential, order-dependent (bias ↓)'. Use neutral grey for the tree boxes; no need for axes."
>
> **After running, verify:**
> - [ ] Left panel shows parallel trees feeding into an average box
> - [ ] Right panel shows trees chained through residuals to a final sum
> - [ ] Subtitles under each panel name the variance-vs-bias distinction


In [ ]:
# Schematic: bagging vs boosting
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bagging — parallel fan
ax = axes[0]
for i in range(5):
    ax.annotate('', xy=(0.55, 0.5), xytext=(0.15, 0.85 - i*0.15),
                arrowprops=dict(arrowstyle='->', color=CLF_COLOR, lw=2))
    ax.text(0.10, 0.85 - i*0.15, f'Tree {i+1}', ha='right', va='center', fontsize=11)
ax.text(0.55, 0.5, 'Average\n(or vote)', ha='center', va='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor=GREY, alpha=0.3))
ax.text(0.85, 0.5, 'Prediction', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor=CLF_COLOR, alpha=0.3))
ax.annotate('', xy=(0.95, 0.5), xytext=(0.65, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.set_title('Bagging (Random Forest) — parallel, independent trees', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

# Boosting — sequential chain
ax = axes[1]
for i in range(5):
    x_pos = 0.10 + i*0.18
    ax.text(x_pos, 0.5, f'T{i+1}', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round', facecolor=REG_COLOR, alpha=0.3))
    if i < 4:
        ax.annotate('', xy=(x_pos + 0.13, 0.5), xytext=(x_pos + 0.05, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(x_pos, 0.30, 'fits\nresiduals\nof T1..T'+str(i) if i > 0 else 'fits\nfull data',
            ha='center', va='center', fontsize=8, color=GREY)
ax.text(0.5, 0.78, 'Σ scaled by learning_rate',
        ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor=GREY, alpha=0.3))
ax.set_title('Boosting (Gradient Boosting) — sequential, each tree fixes the previous error',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

plt.tight_layout()
plt.show()


**Reading the output:**

The two diagrams capture the structural difference. Bagging is **parallel and order-independent** — train tree 1 and tree 5 simultaneously, average them at the end, and the order does not matter. Boosting is **sequential and order-dependent** — tree 5 is fit on residuals that trees 1 through 4 produced together, so the order matters and the trees cannot be trained in parallel.

The order dependence is what makes boosting *slower per training run* than a forest of the same size — you cannot parallelize across trees on a multi-core machine, so the wall-clock cost scales linearly with `n_estimators`. It is also what makes boosting *more accurate* on most tabular datasets, because each new tree's job is **specifically defined** (*"fix what the running ensemble still gets wrong"*) rather than the bagging analyst's job of *"be a different tree."* The price you pay for that focused specialization is sensitivity to hyperparameters — Sections 5 through 7 walk through the three main dials (`learning_rate`, `n_estimators`, `max_depth`) and how they interact.

> **A question that often comes up here:** *"if boosting reduces bias and bagging reduces variance, why not always pick boosting?"* Three reasons. First, **sequential dependence**: you cannot fit `n_estimators` boosted trees in parallel on a multi-core machine, so wall-clock training time scales linearly with `n_estimators` while a random forest of the same size fits in roughly `O(1)` wall-clock time given enough cores. Second, **hyperparameter sensitivity**: a random forest works well at sklearn's defaults, but a GBM at the wrong `learning_rate × n_estimators × max_depth` combination can underfit or overfit dramatically — Section 6's joint-tuning grid is the antidote. Third, **overfitting risk**: more trees never hurt a forest, but more trees in a GBM *can* and *do* overfit (Section 7's `staged_predict` diagnostic shows this directly). The right framing for nb14: bagging and boosting are complementary, and the CI-overlap rule decides which one ships per business case.

**Key takeaway.** Bagging cancels variance by averaging independent trees; boosting reduces bias by stacking trees that each correct the previous error. The two are complementary — nb14 will compare both, under identical CV folds, alongside the linear references from nb09. Neither is universally better; the data decides.

---

## 4. Baseline Gradient Boosting — Default Hyperparameters on Both Cases

Before you tune anything, run the GBM at sklearn's defaults and ask the only question that matters under the CI-overlap rule: **does it beat the best models the course has so far?** Those *"best so far"* picks are nb12's two verdicts, and they set the displacement bar nb13's GBM has to clear:

- **Classification (Wisconsin Breast Cancer):** the Week-2 reference `LogReg(C=1.0)` from nb09 — confirmed by nb12 §9 as the model that ships for the State Health Department's screening tool.
- **Regression (California Housing):** the random forest `RandomForestRegressor(n_estimators=50, max_features='sqrt')` from nb12 §5 — confirmed by nb12 §9 as the model that ships for HomeValue Analytics' price-prediction tool.

The sklearn defaults for `GradientBoostingClassifier` and `GradientBoostingRegressor` are:

- `n_estimators=100` — one hundred boosting iterations
- `learning_rate=0.1` — each tree's contribution is shrunk by 0.1 before being added to the running ensemble
- `max_depth=3` — each tree is intentionally shallow (the *weak learner* idea)
- `loss='log_loss'` for classification, `'squared_error'` for regression

These defaults are surprisingly competitive on most tabular datasets — but *competitive does not always mean better*. The CV-CI dot plot on each panel reports the default GBM next to nb12's pick and the Week-2 reference, so you can read displacement (or not) directly off the bars.

> 💡 **Gemini Prompt:** "Run a three-candidate comparison per case at 5-fold CV. **Classification:** GradientBoostingClassifier(random_state=474) at defaults vs RandomForestClassifier(n_estimators=50, random_state=474) — nb12's pick — vs reference_clf (the LogReg(C=1.0) pipeline). Use cv_clf with scoring='roc_auc'. **Regression:** GradientBoostingRegressor(random_state=474) at defaults vs RandomForestRegressor(n_estimators=50, max_features='sqrt', random_state=474) — nb12's pick — vs reference_reg (the OLS pipeline). Use cv_reg with scoring='r2'. Print mean ± SD ± 95% half-width (df=4) for each candidate, then render two side-by-side CV-CI dot plots."
>
> **After running, verify:**
> - [ ] Classification panel shows three candidates with overlapping CIs (all near 0.98–0.99)
> - [ ] Regression panel shows GBM-default below RF(50, sqrt) by a small but measurable margin
> - [ ] All `cross_val_score` calls use `n_jobs=-1`
> - [ ] CIs are computed with `stats.t.ppf(0.975, df=4)` and `np.sqrt(5)`


In [ ]:
# Baseline default GBM vs nb12 winners — identical 5-fold CV folds as nb12.
gbm_clf = GradientBoostingClassifier(random_state=RANDOM_SEED)
gbm_reg = GradientBoostingRegressor(random_state=RANDOM_SEED)
forest_clf = RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1)                                   # nb12 pick
forest_reg = RandomForestRegressor(n_estimators=50, max_features='sqrt', random_state=RANDOM_SEED, n_jobs=-1)               # nb12 pick

clf_scores = {
    'Week-2 reference: LogReg(C=1.0)':          cross_val_score(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (n=50, nb12 pick)':          cross_val_score(forest_clf,    X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Gradient Boosting (defaults)':             cross_val_score(gbm_clf,       X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}
reg_scores = {
    'Week-2 reference: OLS':                    cross_val_score(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Random Forest (n=50, sqrt, nb12 pick)':    cross_val_score(forest_reg,    X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Gradient Boosting (defaults)':             cross_val_score(gbm_reg,       X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)
def _print_ci_summary(scores_dict, title):
    rows = []
    for name, s in scores_dict.items():
        mean = float(s.mean()); sd = float(s.std(ddof=1))
        half_w = t_crit * sd / np.sqrt(k)
        rows.append({'model': name, 'mean': mean, 'sd': sd, 'half_w': half_w,
                     'ci_low': mean - half_w, 'ci_high': mean + half_w})
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))

_print_ci_summary(clf_scores, "=== CLASSIFICATION (5-fold CV ROC-AUC) — default GBM vs nb12 picks ===")
print()
_print_ci_summary(reg_scores, "=== REGRESSION (5-fold CV R²) — default GBM vs nb12 picks ===")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_cv_ci(clf_scores, 'ROC-AUC', 'Classification — default GBM vs nb12 picks',  axes[0], color=CLF_COLOR)
plot_cv_ci(reg_scores, 'R²',      'Regression — default GBM vs nb12 picks',      axes[1], color=REG_COLOR)
fig.suptitle('Default GBM vs the nb12 champions — has GBM earned displacement at defaults?',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The two business cases give two very different answers about whether default GBM has earned a seat at the candidate table.

**Classification case (Wisconsin Breast Cancer).** Default GBM lands at CV ROC-AUC ≈ **0.981**, the random forest at ≈ **0.987**, and the Week-2 LogReg(C=1.0) at ≈ **0.994**. All three CIs overlap heavily on this small, mostly-linearly-separable dataset. By the CI-overlap rule, **no model has earned displacement of the LogReg reference** — default GBM ties everything else, and the simpler model wins by parsimony. The State Health Department's pick is still LogReg(C=1.0). Default GBM has not yet made the case for itself on the classification side; Sections 5 through 7 will probe whether tuning changes that, and Section 8 will run the formal comparison.

**Regression case (California Housing).** Here default GBM **does not match** the nb12 forest. Forest CV R² ≈ **0.803**, default GBM ≈ **0.784** — a roughly 2-R²-point mean gap with overlapping but still meaningful CIs (forest CI roughly (0.79, 0.82) vs GBM CI roughly (0.77, 0.80)). Under strict CI-overlap this is a tie, but the mean direction is wrong for shipping default GBM: the random forest has the higher mean *and* the simpler defense. **HomeValue Analytics still ships the forest from nb12.** The mechanical reason for default GBM's slight loss is that sklearn's default `max_depth=3` is too shallow for California Housing — `RandomForestRegressor` lets each tree grow deep, so the bagged ensemble captures more structure than the shallow boosted ensemble. **Section 8 will fix this by tuning `max_depth=5`**, at which point GBM closes the gap with the forest.

Default GBM does clear OLS on regression decisively (R² 0.78 vs 0.59, CIs nowhere near overlapping), so the candidate has earned its place above the Week-2 floor — it just has not yet matched the bagged ensemble.

> **A question that often comes up here:** *"if default GBM does not beat the nb12 forest on regression, why was it worth fitting?"* Two reasons. First, **the comparison itself is the discipline**: every new model in nb13's candidate roster must clear the nb12 picks (or tie them under CI-overlap) to earn the right to ship. Default GBM showing that it merely *ties* the random forest on regression is a real finding — it says GBM is *worth tuning further* but not worth shipping out-of-the-box. Second, **default GBM is the starting point** — Sections 5, 6, and 7 will tune it on three dials (`learning_rate`, `n_estimators`, `max_depth`), and Section 8 will run the formal five-candidate comparison that nb14 inherits.

**Key takeaway.** Default GBM ties everything on classification (all CIs overlap → LogReg still wins by parsimony) and *loses by mean* to the nb12 forest on regression (CIs overlap but RF has the higher mean). Default GBM is the *floor* for the tuning work in Sections 5 through 7 — the goal is not to ship the default but to find out whether a tuned GBM can clear nb12's picks under strict CI-overlap.

---

## 5. Learning Rate Trade-off — Slow Learning Beats Fast Learning

Think of `learning_rate` as a **volume knob on each tree's vote**. With `learning_rate=0.1` and 100 trees, every tree casts a 10% vote and the ensemble averages 100 quiet voices into one prediction. With `learning_rate=0.5` and 100 trees, every tree casts a 50% vote — the ensemble converges in a hurry but is more likely to overshoot, because a single noisy tree can swing the running prediction by a lot.

The trade-off in one sentence: **a smaller `learning_rate` needs more `n_estimators` to reach the same accuracy, but usually ends up at a *better* peak.** Slow, steady corrections smooth out individual-tree noise; loud, fast corrections tend to overcorrect. The sweep below holds `n_estimators=100` fixed and tries five learning rates (`0.01, 0.05, 0.1, 0.2, 0.5`) so the shape of the trade-off is visible on both business cases side-by-side. The State Health Department wants to know if dialing this knob meaningfully changes tumor-classification accuracy; HomeValue Analytics wants to know how much R² they leave on the table by accepting the default.


> 💡 **Gemini Prompt:** "Sweep learning_rate in [0.01, 0.05, 0.1, 0.2, 0.5] at fixed n_estimators=100 on both cases using 5-fold CV — cv_clf with scoring='roc_auc' for classification, cv_reg with scoring='r2' for regression. For each setting record the CV mean and SD across folds. Render a 1×2 plot of CV mean with error bars (one SD) and a log-scaled x-axis for learning_rate."
>
> **After running, verify:**
> - [ ] Both panels use a log-scale x-axis
> - [ ] Classification scores cluster in 0.96–0.99 across the sweep (case is forgiving)
> - [ ] Regression rises from R² ~0.50 at lr=0.01 to ~0.80 at lr=0.20 (case is sensitive)
> - [ ] All `cross_val_score` calls use `n_jobs=-1` and 5-fold CV


In [ ]:
# Learning rate sweep — 5-fold CV at fixed n_estimators=100.
lr_grid = [0.01, 0.05, 0.1, 0.2, 0.5]

clf_lr_means, clf_lr_sds = [], []
reg_lr_means, reg_lr_sds = [], []
for lr in lr_grid:
    s_c = cross_val_score(GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=RANDOM_SEED),
                          X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1)
    clf_lr_means.append(s_c.mean()); clf_lr_sds.append(s_c.std(ddof=1))
    s_r = cross_val_score(GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=RANDOM_SEED),
                          X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1)
    reg_lr_means.append(s_r.mean()); reg_lr_sds.append(s_r.std(ddof=1))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.errorbar(lr_grid, clf_lr_means, yerr=clf_lr_sds, marker='o', linewidth=2, capsize=6, color=CLF_COLOR)
ax.set_xscale('log')
ax.set_xlabel('learning_rate (log scale)'); ax.set_ylabel('5-fold CV ROC-AUC')
ax.set_title('Classification — learning rate sweep at n=100', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.errorbar(lr_grid, reg_lr_means, yerr=reg_lr_sds, marker='o', linewidth=2, capsize=6, color=REG_COLOR)
ax.set_xscale('log')
ax.set_xlabel('learning_rate (log scale)'); ax.set_ylabel('5-fold CV R²')
ax.set_title('Regression — learning rate sweep at n=100', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

fig.suptitle('learning_rate × n_estimators are coupled — 100 trees needs lr ≥ 0.05 to fit',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The two cases react to the learning-rate knob very differently — and that contrast is itself the lesson.

**Classification case — State Health Department (Wisconsin Breast Cancer).** Across the full sweep, CV ROC-AUC sits in a narrow **0.965–0.987** band. The slowest setting (`lr=0.01`) underfits — 100 trees casting 1% votes each only deliver one full tree's worth of signal, and the score drops slightly because of it. From `lr=0.05` upward every score is statistically tied; `lr=0.05` peaks at about **0.985**, fractionally above the default (`lr=0.1` at ~0.981), but well inside the fold-to-fold confidence interval. **For the Health Department's classifier, the learning-rate knob barely moves the needle.** Wisconsin is a near-ceiling problem (LogReg already sits at ROC-AUC ~0.993), so there is little structure left for the boosting ensemble to learn no matter how this knob is set.

**Regression case — HomeValue Analytics (California Housing).** Here the knob is consequential. At `lr=0.01` CV R² is only about **0.50** — the ensemble has nowhere near enough total contribution to fit the structure in 12,384 census tracts. At `lr=0.05` it reaches about **0.75**, at `lr=0.10` about **0.78**, at `lr=0.20` about **0.80**, and `lr=0.50` lands around **0.80** with a slight wobble back down. **The sweet spot at fixed `n_estimators=100` is around `lr=0.20`** — a roughly **30 R²-point swing** versus the slowest setting. In business terms, that is the difference between a price model so coarse it cannot support neighborhood-level decisions and one that captures most of the price variance. To match that peak with a smaller learning rate, the analyst has to compensate with more trees — which is exactly what Section 6's joint grid measures.

> **A question that often comes up here:** *"why is classification so flat while regression cares so much about learning rate?"* Two reasons. First, **ceiling effects.** Wisconsin is already near the top of the score; there is barely any residual signal left for any model to extract, so the knob looks identical at every setting. California Housing has rich non-linear structure that takes serious learning-rate × n_estimators budget to capture. Second, **what the loss function rewards.** Log-loss flattens out fast once the model is confidently right; MSE keeps rewarding tighter and tighter price fits all the way down. Tuning matters more wherever the loss curve still has room to fall.

**Key takeaway.** `learning_rate` and `n_estimators` come as a pair — never tune one without the other. On forgiving classification problems (near-ceiling tasks like Wisconsin), defaults are fine. On rich regression problems (the HomeValue case), the choice can drive a 30-point R² swing. Section 6's joint 3×3 grid is the right way to pick both at once.

---


## 6. n_estimators × learning_rate — Joint Tuning Grid

Section 5 made the case that one knob alone is misleading — its effect depends on how many trees the ensemble is allowed to build. Think of it as a road trip: **a slow, steady driver who logs more hours arrives in roughly the same place as a faster driver who logs fewer.** The 3×3 heatmap below pairs `n_estimators ∈ {50, 100, 200}` with `learning_rate ∈ {0.05, 0.1, 0.2}` and runs each of the nine combinations through 5-fold CV on both cases. The pattern to look for is a **diagonal of equivalent performers**: low rate × many trees ≈ high rate × few trees. The off-diagonal corners (small × small, or large × large) are wasteful — small-small never builds enough total signal to fit, and large-large overshoots and chases fold-to-fold noise.

For the **State Health Department**, this grid answers "is there *any* hyperparameter pair on the GBM that beats the LogReg already shortlisted in nb12?" For **HomeValue Analytics**, it answers "where on the grid is the best regression R² a tuned GBM can extract, and is it worth the parsimony cost over the RF currently leading the regression case?"


> 💡 **Gemini Prompt:** "Run a 3×3 grid sweep — n_estimators in [50, 100, 200] crossed with learning_rate in [0.05, 0.1, 0.2] — for both cases using 5-fold CV. Classification: GradientBoostingClassifier(random_state=474) with cv_clf and scoring='roc_auc'. Regression: GradientBoostingRegressor(random_state=474) with cv_reg and scoring='r2'. For each combination record the CV mean; render two side-by-side heatmaps (classification on the left in a Blues colormap; regression on the right in an Oranges colormap) with the CV mean annotated in each cell."
>
> **After running, verify:**
> - [ ] Both heatmaps are 3 rows (`learning_rate`) by 3 columns (`n_estimators`)
> - [ ] Each cell shows the CV mean to 4-decimal precision
> - [ ] Classification heatmap values cluster around 0.98 with little spread
> - [ ] Regression heatmap shows a clear diagonal pattern with values rising from ~0.67 (top-left) to ~0.81 (bottom-right)


In [ ]:
# Joint sweep n_estimators × learning_rate — 5-fold CV on both cases.
n_grid = [50, 100, 200]
lr_grid_joint = [0.05, 0.1, 0.2]

clf_mat = np.zeros((len(lr_grid_joint), len(n_grid)))
reg_mat = np.zeros((len(lr_grid_joint), len(n_grid)))
for i, lr in enumerate(lr_grid_joint):
    for j, n in enumerate(n_grid):
        clf_mat[i, j] = cross_val_score(
            GradientBoostingClassifier(n_estimators=n, learning_rate=lr, random_state=RANDOM_SEED),
            X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1).mean()
        reg_mat[i, j] = cross_val_score(
            GradientBoostingRegressor(n_estimators=n, learning_rate=lr, random_state=RANDOM_SEED),
            X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1).mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, mat, title, cmap in [
    (axes[0], clf_mat, 'Classification — 5-fold CV ROC-AUC', 'Blues'),
    (axes[1], reg_mat, 'Regression — 5-fold CV R²',          'Oranges'),
]:
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            ax.text(j, i, f'{mat[i, j]:.4f}', ha='center', va='center', fontsize=11,
                    color='black' if mat[i, j] < mat.max() * 0.93 else 'white', fontweight='bold')
    ax.set_xticks(range(len(n_grid))); ax.set_xticklabels([str(n) for n in n_grid])
    ax.set_yticks(range(len(lr_grid_joint))); ax.set_yticklabels([f'lr={lr}' for lr in lr_grid_joint])
    ax.set_xlabel('n_estimators'); ax.set_title(title, fontsize=12, fontweight='bold')

fig.suptitle('Joint n_estimators × learning_rate grid — diagonal is the sweet trajectory',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The two heatmaps tell different stories at different intensities.

**Classification case — State Health Department.** All nine cells land in a narrow **0.978–0.985** band. The diagonal pattern is faintly visible but the differences are tiny — every cell is statistically tied with every other cell once you account for the fold-to-fold CI. **There is no clear "best" GBM configuration on Wisconsin**; pick anywhere in the (`lr=0.05–0.2`, `n=50–200`) box and you land near the same CV mean. The simplest-wins tiebreaker says: **stick with `(lr=0.1, n=100)`** — sklearn's defaults — because no other cell on the grid moves the needle. None of this beats the nb12 LogReg leader (0.9935), so the GBM-vs-LogReg verdict from nb12 still stands for the Health Department's screen.

**Regression case — HomeValue Analytics.** Here the grid is informative. The best cells sit at high `learning_rate` paired with high `n_estimators`: **`(lr=0.20, n=200)` lands at CV R² ≈ 0.811**, and `(lr=0.10, n=200)` is right behind at ≈ **0.804**. The worst cell — `(lr=0.05, n=50)` at ≈ **0.674** — has nowhere near enough budget to fit the structure in 12,384 tracts. The diagonal effect is clear: small rate × many trees ≈ large rate × few trees, both beating the off-diagonal corners by 10–14 R² points. Using the simplest-wins rule on cells that tie at the top: **`(lr=0.1, n=100)`** at ≈ 0.78 is the parsimonious pick; if HomeValue wants the maximum CV mean, `(lr=0.1, n=200)` and `(lr=0.20, n=200)` are both legitimate. Section 7 will refine the n-direction with the `staged_predict` early-stopping diagnostic, and Section 8 will land the final tuned-GBM pick — `(lr=0.1, n=100, max_depth=5)` — which adds depth tuning on top of this grid.

> **A question that often comes up here:** *"if more trees at a slower rate are better, why not just run `lr=0.001` with `n=10,000` and call it done?"* Compute and diminishing returns. At ten thousand trees, GBM's fit time grows linearly while marginal accuracy gain shrinks fast. Past `n_estimators ≈ 500` the CV curve is flat enough that the extra trees buy minutes of wall-clock time and almost nothing in score — Section 7's `staged_predict` plot makes this visually obvious. On a Colab session, the trade-off is the difference between *finishes in time* and *times out during the cross-validation loop* — a real constraint for HomeValue's analyst, not a textbook one.

**Key takeaway.** Tune `learning_rate × n_estimators` together. The diagonal of the joint grid is the sweet trajectory; off-diagonal corners are wasteful. On forgiving classification the grid is essentially flat — defaults are fine. On rich regression the diagonal lifts CV R² by about 14 points over the worst corner.

---


## 7. Overfitting in Boosting — `staged_predict` and Early Stopping

Random forests cannot overfit by adding more trees. Each forest tree is grown on a different bootstrap sample, so additional trees just average out independent noise — the prediction smooths out and stabilizes. Gradient boosting is different. Each new boosting tree is built specifically to **patch the mistakes the running ensemble just made** — what statisticians call the *residuals*. Eventually those residuals reduce to fold-specific quirks — noise that does not generalize. The training loss keeps falling toward zero. The validation loss falls, bottoms out, then **starts to climb** as later trees memorize quirks instead of patterns. The optimal `n_estimators` is exactly where the validation curve bottoms.

`staged_predict` is sklearn's diagnostic for finding that turning point — and it does so cheaply. Fit one GBM with a generously large `n_estimators` (say 300), then **rewind the tape**: at every intermediate tree count, `staged_predict` returns the ensemble prediction *as if you had stopped there*, without refitting. We compute the validation loss at every stage, plot train and validation curves side-by-side, and drop a vertical line at the iteration where validation bottoms. The State Health Department wants to know whether the boosting screen is silently overfitting on a 341-patient training pool; HomeValue Analytics wants to know whether 300 trees is enough budget to capture California's price structure, or whether to request more compute.


> 💡 **Gemini Prompt:** "Fit GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, random_state=474) on X_train_clf, y_train_clf. Iterate over staged_predict_proba on both X_train_clf and X_val_clf to compute log_loss at each of the 300 iterations. Repeat for regression with GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=474), using staged_predict + mean_squared_error on X_train_reg / X_val_reg. Find the iteration that minimizes the validation loss per case. Render a 1×2 plot of train vs val loss across iterations, with a vertical red dashed line at the optimal n_estimators per case."
>
> **After running, verify:**
> - [ ] Both panels show 300 iterations on the x-axis
> - [ ] Train loss falls monotonically toward zero on both cases
> - [ ] Classification: validation loss bottoms around iteration 150–170 then climbs slowly
> - [ ] Regression: validation loss continues to fall through iteration 300 (boundary effect — could push higher with more compute)
> - [ ] Red dashed line marks the per-case optimal n_estimators


In [ ]:
# Use the canonical validation set for the staged-predict held-out curve.
# X_val_* was set aside in Section 3 specifically for one-shot held-out checks like this one.

# Classification: log-loss
gbm_c = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05,
                                   random_state=RANDOM_SEED).fit(X_train_clf, y_train_clf)
train_loss_c = []
val_loss_c   = []
for proba_train, proba_val in zip(gbm_c.staged_predict_proba(X_train_clf),
                                   gbm_c.staged_predict_proba(X_val_clf)):
    train_loss_c.append(log_loss(y_train_clf, proba_train))
    val_loss_c.append(log_loss(y_val_clf, proba_val))
opt_n_c = int(np.argmin(val_loss_c)) + 1

# Regression: MSE
gbm_r = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                  random_state=RANDOM_SEED).fit(X_train_reg, y_train_reg)
train_loss_r = []
val_loss_r   = []
for pred_train, pred_val in zip(gbm_r.staged_predict(X_train_reg),
                                 gbm_r.staged_predict(X_val_reg)):
    train_loss_r.append(mean_squared_error(y_train_reg, pred_train))
    val_loss_r.append(mean_squared_error(y_val_reg, pred_val))
opt_n_r = int(np.argmin(val_loss_r)) + 1

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ax = axes[0]
ax.plot(range(1, 301), train_loss_c, label='Train log-loss', color=GREY, linewidth=2)
ax.plot(range(1, 301), val_loss_c,   label='Val log-loss',   color=CLF_COLOR, linewidth=2)
ax.axvline(opt_n_c, color='red', linestyle='--', label=f'Early stop @ n={opt_n_c}')
ax.set_xlabel('n_estimators'); ax.set_ylabel('log-loss')
ax.set_title('Classification — staged loss curves', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(range(1, 301), train_loss_r, label='Train MSE', color=GREY, linewidth=2)
ax.plot(range(1, 301), val_loss_r,   label='Val MSE',   color=REG_COLOR, linewidth=2)
ax.axvline(opt_n_r, color='red', linestyle='--', label=f'Early stop @ n={opt_n_r}')
ax.set_xlabel('n_estimators'); ax.set_ylabel('MSE')
ax.set_title('Regression — staged loss curves', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Early stopping — train loss → 0; val loss eventually turns up',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 Classification early-stop n_estimators: {opt_n_c}")
print(f"💡 Regression early-stop n_estimators:     {opt_n_r}")


**Reading the output:**

The two curves tell two different stopping stories.

**Classification case — State Health Department (Wisconsin Breast Cancer).** Training log-loss falls monotonically toward zero — by about iteration 100 it is essentially nil. Validation log-loss falls quickly in the first 30–50 iterations, bottoms out at around iteration **~159** (at `learning_rate=0.05`), and then **starts to climb** as later trees memorize quirks specific to the 341-patient training pool. The vertical red line marks the early-stopping point — past it, more trees actively *hurt* generalization. This is the classic boosting-overfitting signature, and on a hospital-scale dataset like Wisconsin it kicks in early. The honest deployment recipe for the Health Department: fit one GBM with `n_estimators=300`, run `staged_predict`, then **refit with `n_estimators=159`** (or whatever the rerun shows) on the full training set.

**Regression case — HomeValue Analytics (California Housing).** Same shape, different scale. Training MSE falls toward zero, validation MSE falls and bottoms out — but at this seed and `learning_rate=0.05` validation MSE **keeps falling all the way to `n_estimators=300`** without turning back up. The vertical red line lands at the right edge of the plot. With **12,384 training tracts** — 36× more rows than Wisconsin — California Housing has much more signal for boosting to extract before noise-fitting kicks in. The boundary-edge optimum is a hint: if HomeValue could afford to extend the sweep to `n=500` or `n=1000`, they might squeeze out a few more R² points. On a Colab time budget, 300 is a sensible stopping point.

> **A question that often comes up here:** *"if early stopping is so important, why not always use sklearn's automatic `n_iter_no_change`?"* You can — `GradientBoostingClassifier(n_iter_no_change=10, validation_fraction=0.1)` automates the protocol in one line. The reason for running `staged_predict` manually here is **visibility**: a stakeholder briefing reads much more credibly when you can show the train-vs-validation crossover on screen instead of asserting that sklearn handled it. In production code, once the protocol is trusted, `n_iter_no_change` is fine and lands on the same `n_estimators` to within a tree or two.

**Key takeaway.** Boosting overfitting is real but easy to diagnose. `staged_predict` traces train and validation loss curve-by-curve, and the validation minimum is the right `n_estimators`. On small near-ceiling problems like Wisconsin, the minimum comes early (~150–170 trees); on richer regression problems like California Housing, the minimum can sit at the boundary of the sweep — meaning more trees might still help if the compute budget allows.

---


## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Tune the Classification GBM

**Task:** Find the best `(n_estimators, learning_rate)` combination for `GradientBoostingClassifier` on Wisconsin breast cancer using a 3×3 grid.

**Instructions:**
1. Sweep `n_estimators ∈ [50, 100, 200]` × `learning_rate ∈ [0.05, 0.1, 0.2]`.
2. For each combination, compute 3-fold CV ROC-AUC mean and SD using `cv_clf_3`.
3. Render as a heatmap with mean values annotated.
4. Apply the one-SE rule.
5. Write 3 short findings on whether the diagonal pattern from Section 6 confirmed itself.

---

> 💡 **Gemini Prompt:** "Grid-search GradientBoostingClassifier(random_state=474) over n_estimators=[50,100,200] × learning_rate=[0.05,0.1,0.2] using 3-fold CV ROC-AUC. Build a 3×3 heatmap with means annotated; star the best cell; apply the one-SE rule."


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune GradientBoostingClassifier over (n_estimators, learning_rate); apply one-SE rule.


## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Tune the Regression GBM

**Task:** Find the best `(n_estimators, learning_rate)` combination for `GradientBoostingRegressor` on California Housing.

**Instructions:**
1. Same 3×3 grid as Exercise 1 (`n ∈ [50, 100, 200]` × `lr ∈ [0.05, 0.1, 0.2]`).
2. 3-fold CV R² with `cv_reg_3`.
3. Heatmap with mean annotations and starred best cell.
4. Convert best CV-RMSE to USD; apply the one-SE rule.

---

> 💡 **Gemini Prompt:** "Grid-search GradientBoostingRegressor(random_state=474) over n_estimators=[50,100,200] × learning_rate=[0.05,0.1,0.2] using 3-fold CV R². Heatmap of CV means; report best CV-RMSE in USD; apply the one-SE rule."


In [ ]:
# YOUR SOLUTION CODE HERE
# Tune GradientBoostingRegressor; report best CV-RMSE in USD.


## 8. Final Comparison — All Models on Both Cases

The closing section is the **direct setup for nb14's selection ceremony**. Five candidates per business case, scored on identical 5-fold CV folds, plotted as CV-CI dot plots side by side. The five candidates trace the analytical story the course has built since nb09:

1. **Week-2 reference** — `LogReg(C=1.0)` for classification, OLS for regression. The linear baseline that survived nb09's tuning sweeps.
2. **nb11's single tree** — `DecisionTreeClassifier(max_depth=3)` for classification, `DecisionTreeRegressor(max_depth=5)` for regression. The single-tree picks from nb11 §7.
3. **nb12's random forest** — `RandomForestClassifier(n_estimators=50)` for classification, `RandomForestRegressor(n_estimators=50, max_features='sqrt')` for regression. The nb12 §5 / §9 picks — and nb12's regression champion.
4. **Default GBM** — `GradientBoostingClassifier(random_state=474)` and `GradientBoostingRegressor(random_state=474)`. The sklearn defaults from §4.
5. **Tuned GBM** — the configurations Sections 5–7 led to:
   - Classification: `(learning_rate=0.05, n_estimators=100, max_depth=3)` — defaults except for a slightly slower learning rate, since §6's grid was nearly flat on Wisconsin.
   - Regression: `(learning_rate=0.1, n_estimators=100, max_depth=5)` — `max_depth=5` is the unlock that lets GBM compete with the random forest on California Housing.

The CI-overlap rule from nb08 decides each case's verdict. The Reading-the-output below names the picks explicitly so you can defend them in writing to the two stakeholder teams.

> 💡 **Gemini Prompt:** "Build a five-candidate CV-CI comparison per case. **Classification:** (a) reference_clf — LogReg(C=1.0) pipeline; (b) DecisionTreeClassifier(max_depth=3, random_state=474); (c) RandomForestClassifier(n_estimators=50, random_state=474); (d) GradientBoostingClassifier(random_state=474); (e) GradientBoostingClassifier(learning_rate=0.05, n_estimators=100, random_state=474). Use cv_clf with scoring='roc_auc'. **Regression:** (a) reference_reg — OLS pipeline; (b) DecisionTreeRegressor(max_depth=5, random_state=474); (c) RandomForestRegressor(n_estimators=50, max_features='sqrt', random_state=474); (d) GradientBoostingRegressor(random_state=474); (e) GradientBoostingRegressor(learning_rate=0.1, n_estimators=100, max_depth=5, random_state=474). Use cv_reg with scoring='r2'. Print mean ± SD ± 95% half-width (df=4) per candidate per case. Render two side-by-side CV-CI dot plots."
>
> **After running, verify:**
> - [ ] Each panel has exactly five candidates labeled in the legend / y-axis
> - [ ] Classification panel: LogReg at the top by mean (~0.994); all four other CIs overlap LogReg
> - [ ] Regression panel: tuned GBM at the top by mean (~0.811); RF just below (~0.803); CIs overlap
> - [ ] Default GBM (clf and reg) sits between RF and the single tree
> - [ ] All `cross_val_score` calls use `n_jobs=-1`


In [ ]:
# Build five candidates per case — match nb11 (tree depths) and nb12 (RF config) picks
# and the §5–§6 tuning verdicts (lr/n/depth for the tuned GBM per case).
clf_compare = {
    'Week-2 reference: LogReg(C=1.0)':            cross_val_score(reference_clf, X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Single Tree (depth=3, nb11)':                cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'Random Forest (n=50, nb12 pick)':            cross_val_score(RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED, n_jobs=-1), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'GBM default (lr=0.1, n=100, d=3)':           cross_val_score(GradientBoostingClassifier(random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
    'GBM tuned (lr=0.05, n=100, d=3)':            cross_val_score(GradientBoostingClassifier(learning_rate=0.05, n_estimators=100, random_state=RANDOM_SEED), X_train_clf, y_train_clf, cv=cv_clf, scoring='roc_auc', n_jobs=-1),
}

reg_compare = {
    'Week-2 reference: OLS':                       cross_val_score(reference_reg, X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Single Tree (depth=5, nb11)':                 cross_val_score(DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'Random Forest (n=50, sqrt, nb12 pick)':       cross_val_score(RandomForestRegressor(n_estimators=50, max_features='sqrt', random_state=RANDOM_SEED, n_jobs=-1), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'GBM default (lr=0.1, n=100, d=3)':            cross_val_score(GradientBoostingRegressor(random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
    'GBM tuned (lr=0.1, n=100, d=5)':              cross_val_score(GradientBoostingRegressor(learning_rate=0.1, n_estimators=100, max_depth=5, random_state=RANDOM_SEED), X_train_reg, y_train_reg, cv=cv_reg, scoring='r2', n_jobs=-1),
}

k = 5
t_crit = stats.t.ppf(0.975, df=k - 1)
def _print_ci_summary(scores_dict, title):
    rows = []
    for name, s in scores_dict.items():
        mean = float(s.mean()); sd = float(s.std(ddof=1))
        half_w = t_crit * sd / np.sqrt(k)
        rows.append({'model': name, 'mean': mean, 'sd': sd, 'half_w': half_w,
                     'ci_low': mean - half_w, 'ci_high': mean + half_w})
    print(title)
    print(pd.DataFrame(rows).to_string(index=False))

_print_ci_summary(clf_compare, "=== CLASSIFICATION (5-fold CV ROC-AUC) — five candidates ===")
print()
_print_ci_summary(reg_compare, "=== REGRESSION (5-fold CV R²) — five candidates ===")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_cv_ci(clf_compare, 'ROC-AUC', 'Classification — five candidates for nb14', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_compare, 'R²',      'Regression — five candidates for nb14',     axes[1], color=REG_COLOR)
fig.suptitle('Five-candidate roster per case — direct setup for nb14\'s selection ceremony',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

Two business cases, two candidate fields of five models each — and two distinct verdicts under the CI-overlap rule from nb08.

**Classification case (Wisconsin Breast Cancer).** Final means by CV ROC-AUC: LogReg(C=1.0) ≈ **0.994**, RF(n=50) ≈ **0.987**, GBM tuned (`lr=0.05`) ≈ **0.985**, GBM default ≈ **0.981**, Single Tree (d=3) ≈ **0.922**. The LogReg reference is at the top by mean, the random forest and both GBM variants cluster closely below it, and the single tree trails by a clear margin. **All three tree-based ensembles (RF, GBM default, GBM tuned) have CIs that overlap LogReg's heavily** — under strict CI-overlap each of those three is a statistical tie with LogReg. The single tree's CI ends at about 0.981, just below LogReg's lower bound of about 0.986, so LogReg is **CI-clear above the single tree** by a thin but real margin. Among the three tied candidates, LogReg is the simplest model by class (linear vs tree ensemble), so parsimony picks LogReg. **Verdict: ship the Week-2 reference, `LogReg(C=1.0)`** for the State Health Department's screening tool. This is the same pick nb12 §9 made; nb13's tuning work did not produce a candidate that earns CI-clear displacement of the linear baseline on this dataset. *That itself* is the right finding on a small, near-ceiling, mostly-linearly-separable dataset.

**Regression case (California Housing).** Final means by CV R²: GBM tuned (`max_depth=5`) ≈ **0.811**, RF(n=50, sqrt) ≈ **0.803**, GBM default ≈ **0.784**, Single Tree (d=5) ≈ **0.613**, OLS ≈ **0.586**. The tuned GBM has the highest mean — but compare the tuned GBM CI (≈ (0.797, 0.825)) to the random forest CI (≈ (0.790, 0.817)). **The two overlap on (0.797, 0.817)** — by strict CI-overlap, the tuned GBM and the random forest are *statistically tied*. Under parsimony the simpler model wins, and between two tree ensembles tied by CI-overlap, the **random forest is the more conservative choice** on three grounds: (1) bagging is *parallel and order-independent*, while gradient boosting is sequential and order-dependent (slower to fit, harder to scale); (2) the random forest is *robust to hyperparameter choices* — nb12 §5 showed any `n_estimators ≥ 50` and several `max_features` values produce statistically tied CV scores, while nb13 §5 here showed GBM's CV R² swinging from 0.50 to 0.80 across `learning_rate` alone; (3) the random forest *does not amplify leaky features* the way GBM does (Section 9). On all three counts the forest is the lower-risk pick when the means are within statistical-tie distance. **Verdict: ship the random forest, `RandomForestRegressor(n_estimators=50, max_features='sqrt')`** for HomeValue Analytics' price-prediction tool. Same pick nb12 §9 made — but with one important new piece of evidence: the tuned GBM has *earned its place* on nb14's candidate roster as a real challenger.

The mechanical lesson on the regression side is what nb12's wrap-up foreshadowed: **`max_depth=5` is the unlock for GBM on California Housing**. Default GBM with `max_depth=3` (R² ≈ 0.784) sits a CI-clear margin below the forest; tuned GBM with `max_depth=5` (R² ≈ 0.811) ties the forest and edges it on mean. The depth unlock is real but not large enough to *clear* the forest's CI — the regression-side winner remains the random forest under strict parsimony.

> **A question that often comes up here:** *"if the tuned GBM ties the forest on regression, why not ship the GBM since its mean is higher?"* Three reasons. First, **CI-overlap is the rule the course has used since nb08** — and the rule says simpler wins ties. Second, **the forest is genuinely the lower-risk model class** in the relevant sense: bagging is parallelizable, robust to hyperparameter choices, and does not amplify leakage — all three properties that gradient boosting lacks. A stakeholder asking *"what would happen if we retrained on next month's data?"* gets a much more reassuring answer from the forest than from a GBM that needs three coupled knobs (`learning_rate × n_estimators × max_depth`) re-tuned each time. Third, **the mean gap is fragile** — 0.811 vs 0.803 is less than 1 R² point, well inside fold-to-fold noise; on a different seed the order could flip. The forest's tighter CI and lower defensibility burden make it the more conservative pick. nb14 will run the same five-candidate field with a sparse linear variant added and apply the same rule.

**Key takeaway.** Five candidates per case, two verdicts under strict CI-overlap, both consistent with nb12's verdicts: **classification ships the Week-2 reference `LogReg(C=1.0)`**; **regression ships `RandomForestRegressor(n_estimators=50, max_features='sqrt')`**. The tuned GBM is now a legitimate challenger on the regression side — its mean is higher than the forest's, but the CIs overlap and parsimony decides the tie.

---

## 9. Wrap-Up — Key Takeaways

**What landed today:**

1. **Boosting is sequential, not parallel.** Each tree fits the residuals of the running ensemble — on regression the literal `y − ŷ`, on classification the gradient of the log-loss which works out to roughly `y − p̂`. The algorithm reduces **bias** (systematic mistakes) rather than variance.

2. **`learning_rate` and `n_estimators` are coupled.** Tune them together — Section 6's joint 3×3 grid is the right tool. The diagonal (small lr × large n, or large lr × small n) is the sweet trajectory; off-diagonal corners are wasteful. On forgiving classification the grid is essentially flat; on rich regression the diagonal lifts CV R² by ~14 points compared to the worst corner.

3. **`staged_predict` is the right early-stopping diagnostic.** Fit one large GBM, iterate over the staged predictions on the canonical `X_val_*` set, find the validation-loss minimum, and refit with that `n_estimators`. On classification (small data) the optimum comes around iteration 150; on regression (large data) the optimum can sit at the boundary of a 300-iteration sweep.

4. **Per-case picks — defensible under strict CI-overlap.**
   - **Classification (Wisconsin Breast Cancer — State Health Department):** ship the Week-2 reference `LogReg(C=1.0)`. All three tree-ensemble candidates (random forest, default GBM, tuned GBM) have CIs that overlap LogReg's, and LogReg is CI-clear above the single tree — statistical tie among the upper four → simpler model wins by parsimony. Same pick as nb12 §9.
   - **Regression (California Housing — HomeValue Analytics):** ship the random forest `RandomForestRegressor(n_estimators=50, max_features='sqrt')`. The tuned GBM (`lr=0.1, n=100, d=5`) has the highest mean (≈ 0.811) but its CI overlaps the forest's CI (≈ 0.803) — statistical tie → simpler model class wins. The mechanical lesson: `max_depth=5` is the unlock that gets GBM into a tie with the forest, but the tie does not break in GBM's favor. Same pick as nb12 §9, with GBM now a real challenger on the candidate roster.

5. **Boosting amplifies leaky features more aggressively than any algorithm in the course — defend against it.** Because every iteration fits the residuals, a leaky column (one engineered from future data, target data, or held-out data) drives the residuals to zero in the first few trees, and the model becomes `y = leaky_feature` plus tiny corrections. On a held-out test set or in production where the leaky column is missing or noisy, the model collapses. The defence is the four-method importance check from nb12 §7: drop or null suspect features one at a time and refit (CV collapse means the feature was carrying signal it should not have); inspect the top-ranked feature in the four-method heatmap (derived fields with names like `_at_t+1`, `_after_`, `_total_lifetime_`, or anything engineered from the label are the usual suspects); and audit the pipeline boundary for any column engineered from future data, target data, or held-out data. The same recipe protects you on every tabular dataset; the urgency just goes up with boosting.

**Bridge to nb14 — the Selection Ceremony:**

You now have the full candidate roster per business case — five models, scored on identical 5-fold CV folds — that nb14 will turn into a formal selection ceremony with a written champion memo and a single, blind opening of the locked test set.

The candidate field per case is exactly what you produced in Section 8:

- **Classification:** Week-2 LogReg(C=1.0), single tree (d=3), random forest (n=50), default GBM, tuned GBM (lr=0.05, n=100, d=3). nb14 will add a regularized linear variant (LogReg L1) to make it a five-vs-five field.
- **Regression:** Week-2 OLS, single tree (d=5), random forest (n=50, sqrt), default GBM, tuned GBM (lr=0.1, n=100, d=5). nb14 will add a regularized linear variant (Lasso).

nb14 does three things you have not yet seen as a single workflow: (a) **declare the comparison protocol in writing before looking at any results** (no model-shopping after seeing the numbers); (b) **write a champion-selection memo** with a primary metric, runner-up CI-overlap test, and stakeholder-language justification; (c) **open the locked test set exactly once per business case** and pronounce an INSIDE / ABOVE / BELOW verdict against the champion's CV CI.

> **A question that often comes up at this point:** *"does gradient boosting always beat random forest?"* On most tabular datasets, yes — by a small margin once `learning_rate × n_estimators × max_depth` are all tuned. But the surprise nb13 surfaces is that **even the tuned GBM at the unlocked `max_depth=5` only *ties* the random forest on California Housing under strict CI-overlap** — the mean is higher but the CIs overlap, and parsimony decides the tie. On Wisconsin breast cancer no tree-based candidate clears the Week-2 LogReg. nb13's job is not to crown a new winner but to **add a defensible challenger to nb14's candidate roster**; nb14's job is to run the ceremony that formally adjudicates.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (GBM tuning, classification) and Exercise 2 (GBM tuning, regression).
2. **Run All Cells** — `Runtime → Run all` to ensure every cell executes without error.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 13 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise solutions produce a tuning heatmap with the chosen combination starred
- [ ] The staged-loss curve renders for both cases
- [ ] All figures render (none broken)

### Next Step:

- **Notebook 14** — Model Selection Protocol + Test-Set Opening Ceremony (Day 14)

---

<center>

**Thank you!**

</center>